# 🔄 Conversión de Gemma a GGUF para Termux/Android

Este notebook convierte modelos Gemma desde su formato original (SafeTensors) a GGUF para usar con llama.cpp en Termux.

## 📋 Opciones de Modelos

| Modelo | Repositorio Original | Tamaño FP16 | Tamaño GGUF Q4_K_M |
|--------|---------------------|-------------|-------------------|
| Gemma 4 E2B IT | `google/gemma-4-e2b-it` | ~4 GB | ~1.5 GB |
| Gemma 4 E4B IT | `google/gemma-4-e4b-it` | ~8 GB | ~2.6 GB |
| Gemma 4 4B IT | `google/gemma-4-4b-it` | ~8 GB | ~2.5 GB |
| Gemma 4 12B IT | `google/gemma-4-12b-it` | ~24 GB | ~7 GB |

## 🚀 Instrucciones

1. Selecciona el modelo a convertir (celda 2)
2. Ejecuta todas las celdas en orden
3. Descarga el archivo GGUF generado
4. Transfiérelo a tu dispositivo Android en `~/gemma-termux-app/models/`

## 1️⃣ Instalación de Dependencias

In [ ]:
# Instalar dependencias necesarias
!pip install -q transformers huggingface_hub torch sentencepiece protobuf

# Clonar llama.cpp para las herramientas de conversión
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git

# Instalar requisitos de llama.cpp
!pip install -q -r llama.cpp/requirements.txt

print("✅ Dependencias instaladas correctamente")

## 2️⃣ Configuración del Modelo

**Opciones de modelos Gemma 4:**

- `google/gemma-4-e2b-it` - 2B params (especializado, ligero)
- `google/gemma-4-e4b-it` - 4B params (especializado, recomendado)
- `google/gemma-4-4b-it` - 4B params (estándar)
- `google/gemma-4-12b-it` - 12B params (grande, requiere más RAM)

In [ ]:
# ============================================
# CONFIGURACIÓN - MODIFICA ESTO
# ============================================

# Modelo a convertir (descomenta el que quieras usar)
MODEL_ID = "google/gemma-4-e4b-it"  # @param ["google/gemma-4-e2b-it", "google/gemma-4-e4b-it", "google/gemma-4-4b-it", "google/gemma-4-12b-it"]

# Nivel de cuantización (recomendado: q4_k_m)
# Opciones: q4_0, q4_k_m, q5_k_m, q6_k, q8_0, f16
QUANTIZATION = "q4_k_m"  # @param ["q4_0", "q4_k_m", "q5_k_m", "q6_k", "q8_0", "f16"]

# Nombre del archivo de salida (opcional, dejar vacío para auto-generar)
CUSTOM_OUTPUT_NAME = ""  # @param {type:"string"}

# ============================================
# NO MODIFICAR DESDE AQUÍ
# ============================================

print(f"📦 Modelo seleccionado: {MODEL_ID}")
print(f"🔧 Cuantización: {QUANTIZATION}")
print(f"💾 Espacio en disco disponible:")
!df -h /content | tail -1

## 3️⃣ Autenticación con Hugging Face

Los modelos Gemma requieren aceptar los términos en Hugging Face.

1. Ve a: https://huggingface.co/google/gemma-4-e4b-it
2. Inicia sesión y acepta los términos
3. Genera un token de acceso: https://huggingface.co/settings/tokens
4. Ejecuta la siguiente celda e introduce tu token

In [ ]:
from huggingface_hub import login

# Login (se te pedirá el token)
login()

print("✅ Autenticación completada")

## 4️⃣ Descarga del Modelo Original

In [ ]:
from huggingface_hub import snapshot_download
import os

# Directorio de descarga
download_dir = f"/content/{MODEL_ID.split('/')[-1]}"
os.makedirs(download_dir, exist_ok=True)

print(f"⬇️ Descargando {MODEL_ID}...")
print(f"📁 Destino: {download_dir}")
print("⏳ Esto puede tardar varios minutos...")
print()

# Descargar el modelo
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=download_dir,
    local_dir_use_symlinks=False,
    resume_download=True
)

print()
print(f"✅ Modelo descargado en: {download_dir}")

# Verificar archivos
print("\n📄 Archivos descargados:")
!ls -lh {download_dir} | head -20

## 5️⃣ Conversión a GGUF

In [ ]:
import subprocess
import sys

# Generar nombre de salida
if CUSTOM_OUTPUT_NAME:
    output_file = f"/content/{CUSTOM_OUTPUT_NAME}"
else:
    model_name = MODEL_ID.split('/')[-1]
    output_file = f"/content/{model_name}-{QUANTIZATION.upper()}.gguf"

print(f"🔄 Iniciando conversión...")
print(f"   Entrada: {download_dir}")
print(f"   Salida: {output_file}")
print(f"   Tipo: {QUANTIZATION}")
print()

# Comando de conversión
convert_cmd = [
    sys.executable,
    "llama.cpp/convert_hf_to_gguf.py",
    download_dir,
    "--outfile", output_file,
    "--outtype", QUANTIZATION
]

print("Ejecutando:", " ".join(convert_cmd))
print()

# Ejecutar conversión
result = subprocess.run(convert_cmd, capture_output=False, text=True)

if result.returncode == 0:
    print()
    print("✅ Conversión completada exitosamente!")
else:
    print()
    print("❌ Error en la conversión")
    if result.stderr:
        print("Error:", result.stderr)

## 6️⃣ Verificación del Archivo Generado

In [ ]:
import os

# Verificar que el archivo existe
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file)
    file_size_gb = file_size / (1024**3)
    
    print("✅ Archivo GGUF generado correctamente")
    print(f"📁 Ubicación: {output_file}")
    print(f"📊 Tamaño: {file_size_gb:.2f} GB ({file_size:,} bytes)")
    
    # Información adicional
    print("\n📋 Información del archivo:")
    !ls -lh {output_file}
else:
    print("❌ Error: No se encontró el archivo de salida")
    print("Archivos en /content:")
    !ls -lh /content/*.gguf 2>/dev/null || echo "No hay archivos .gguf"

## 7️⃣ Descarga del Archivo

### Opción A: Descarga directa desde Colab

Ejecuta la siguiente celda y se descargará automáticamente el archivo.

In [ ]:
from google.colab import files

if os.path.exists(output_file):
    print(f"⬇️ Descargando {os.path.basename(output_file)}...")
    files.download(output_file)
else:
    print("❌ No se encontró el archivo para descargar")

### Opción B: Subir a Google Drive

Si el archivo es muy grande para descargar directamente, súbelo a Drive:

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copiar a Drive
if os.path.exists(output_file):
    drive_path = f"/content/drive/MyDrive/{os.path.basename(output_file)}"
    !cp {output_file} {drive_path}
    print(f"✅ Archivo copiado a Drive: {drive_path}")
else:
    print("❌ No se encontró el archivo")

### Opción C: Subir a Hugging Face Hub

Comparte tu modelo cuantizado con la comunidad:

In [ ]:
# Descomenta para subir a Hugging Face
# from huggingface_hub import HfApi
# api = HfApi()
# 
# repo_id = "tu-usuario/gemma-4-e4b-it-gguf"  # Modifica esto
# api.create_repo(repo_id, exist_ok=True)
# api.upload_file(
#     path_or_fileobj=output_file,
#     path_in_repo=os.path.basename(output_file),
#     repo_id=repo_id
# )
# print(f"✅ Subido a: https://huggingface.co/{repo_id}")

## 📱 Instalación en Android/Termux

Una vez descargado el archivo `.gguf`, transfiérelo a tu dispositivo Android:

```bash
# En Termux, crear directorio de modelos
mkdir -p ~/gemma-termux-app/models/

# Transferir el archivo (vía USB, FTP, o cloud)
# Coloca el archivo .gguf en ~/gemma-termux-app/models/

# Renombrar para facilitar
mv ~/gemma-termux-app/models/gemma-4-*-Q4_K_M.gguf ~/gemma-termux-app/models/gemma-default.gguf

# Ejecutar la app
gemma
```

## 🔧 Solución de Problemas

### Error: "Gated repo"
- Asegúrate de haber aceptado los términos en Hugging Face
- Verifica que tu token tenga permisos de lectura

### Error: "Out of memory"
- Usa un modelo más pequeño (E2B en lugar de E4B)
- O usa cuantización más agresiva (q4_0 en lugar de q4_k_m)

### Archivo muy grande para descargar
- Usa la Opción B (Google Drive)
- O la Opción C (Hugging Face Hub)

## 📚 Recursos Adicionales

- [Gemma en Hugging Face](https://huggingface.co/google)
- [llama.cpp](https://github.com/ggerganov/llama.cpp)
- [GGUF Format](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md)